# Emerging Technologies Assessment

# Introduction
The following notebook examines and demonstrates the **Deutsch-Jozsa Algorithm**, one of the earliest examples of a quantum algorithm outperforming its classical counterpart 

The Deutsch-Jozsa Algorithm is a quantum algorithm that determines whether (in this case) a Boolean function is either **constant** or **balanced** with just a single query to the function. Comparing this to classical computing where it requires $2^{n-1} + 1$ queries in the worst case for a function with $n$ input bits.

For example in classical computing for a 4-input function, the worst case is 9 queries. But for the Deutsch–Jozsa algorithm it can do it in 1.

This advantage comes from two key quantum phenomena:
- **Superposition** — a qubit can exist in both states ($|0\rangle$ and $|1\rangle$) simultaneously until measured. This allows the algorithm to evaluate the function on all inputs at once, rather than one at a time.
- **Interference** — quantum states can interact with each other, enhancing the probability of correct outcomes while cancelling out incorrect ones. The algorithm is structured so this interference directly reveals whether the function is constant or balanced.

## Notebook overview:
| Problem | Topic |
|---------|-------|
| 1 | Generating random constant and balanced Boolean functions |
| 2 | Determining function type classically, and analysing the cost |
| 3 | Building quantum oracles for single-input Boolean functions |
| 4 | Implementing Deutsch's algorithm in Qiskit |
| 5 | Scaling up to the full Deutsch–Jozsa algorithm for 4-input functions |

# Imports


In [ ]:
# Used to generate random numbers for the boolean functions
import random

# Used for array manipulations and mathematical operations
import numpy as np

# Quantum computing libraries for building and simulating quantum circuits
import qiskit
import qiskit_aer as aer

# Used for creating combinations of input variables for the boolean functions
from itertools import product

print("All necessary libraries have been imported successfully.")

# Problem 1: Generating Random Boolean Functions
## Context
The [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) works with a specific type of function: one that takes Boolean inputs and returns a single Boolean output. Each function is guaranteed to be either **constant** or **balanced**.

### Boolean Functions
A [Boolean function](https://realpython.com/python-boolean/) takes `True`/`False` inputs and returns a single `True`/`False` output. In this problem, we work with functions that take **four** Boolean inputs, giving $2^4 = 16$ possible input combinations.

### Constant vs Balanced

| Type | Meaning |
|---|---|
| **Constant** | Returns the same value (`True` or `False`) for all 16 inputs |
| **Balanced** | Returns `True` for exactly 8 inputs, `False` for the other 8 |

There are exactly **2 constant functions** (always-True and always-False). There are $\binom{16}{8} = 12{,}870$ different balanced functions.

The Deutsch–Jozsa algorithm can identify which type a function is using just 1 quantum query, whereas classically we may need up to 9 queries for a 4-input function.

### Truth Tables
A **truth table** is a simple way to represent a Boolean function. It lists all possible inputs and their outputs.

For our 4-input function, we store the truth table as a list of 16 values:

```python
truth_table = [True, False, True, True, False, ..., False]
              # index 0    1      2     3     4         15
```

To look up the output for a given input, we convert the 4 Boolean inputs into a number from 0 to 15 using **bit shifting**:

```python
index = (a << 3) | (b << 2) | (c << 1) | d
```

This converts each Boolean input (True=1, False=0) into a single index. For example, inputs `(True, False, True, False)` become index 10.

Then we simply return `truth_table[index]` to get the output.

### Implementation

The function `random_constant_balanced()` will randomly produce either a constant or a balanced Boolean function. This will serve as the test data generator for the rest of the problems.

Approach:
1. Randomly decide: will this be a constant or balanced function?
2. Build a truth table (list of 16 Boolean values):
   - If constant: all 16 values are the same
   - If balanced: 8 values are `True`, 8 are `False` (shuffled randomly)
3. Create a function that converts inputs to an index and looks up the answer in the truth table
4. Attach metadata to the function so we can inspect what type was created

In [ ]:
def random_constant_balanced():
    """
    Returns a randomly chosen Boolean function that is either constant or balanced.

    The returned function takes four Boolean arguments (a, b, c, d)
    and returns a single Boolean output.

    - Constant: always returns the same value for all 16 inputs.
    - Balanced: returns True for exactly 8 of the 16 inputs.

    Returns:
        A callable f(a, b, c, d) -> bool with attached metadata.
    """
    # Choose randomly between 'constant' and 'balanced'
    is_constant = random.choice([True, False])

    # Build the truth table for the function
    if is_constant:
        # Constant if all 16 outputs are the same
        output_value = random.choice([True, False])
        truth_table = [output_value] * 16
    else:
        # Balanced: 8 True and 8 False outputs
        truth_table = [True] * 8 + [False] * 8
        random.shuffle(truth_table)  # Shuffle to randomize the order

    # Function to use the truth table
    def f(a, b, c, d):
        """
        Evaluate the function for inputs (a, b, c, d).
        Converts inputs to an index and looks up the answer.
        """
        # Convert 4 boolean inputs to an index (0-15)
        index = (a << 3) | (b << 2) | (c << 1) | d
        return truth_table[index]
        
    # Attach metadata to the function
    f.truth_table = truth_table
    f.is_constant = is_constant
    
    return f

### Code Explanation

The function randomly creates either a **constant** or **balanced** function.

First, it flips a metaphorical coin to decide which type to create. Then it builds a truth table, a list of 16 Boolean values.

For a **constant** function, all 16 values are the same (either all `True` or all `False`). For a **balanced** function, it creates 8 `True` and 8 `False` values, then shuffles them randomly.

Next, it defines the function `f(a, b, c, d)` which converts the four Boolean inputs into an index (0–15) using bit shifting, then looks up the answer in the truth table.

Finally, it attaches metadata, the truth table itself and a flag indicating whether the function is constant, so we can inspect the function after creating it.

### Testing
Verify if the function works by evaluating it on all 16 inputs, and checking whether it is constant (all outputs being the same) or balanced (8 True, 8 False).

In [ ]:
# Generate a random Boolean function
f = random_constant_balanced()

# Test with all 16 combinations of 4-bit inputs
print("Testing the function on all 16 4-bit inputs:\n")
print("Input (a, b, c, d) → Output")
print("-" * 40)

outputs = []
for a in [False, True]:
    for b in [False, True]:
        for c in [False, True]:
            for d in [False, True]:
                result = f(a, b, c, d)
                outputs.append(result)
                
                # Output as binary 1 for True and 0 for False
                binary_output = f"{int(a)}{int(b)}{int(c)}{int(d)}"
                print(f"{binary_output} = {result}")

In [ ]:
# Check the True outputs to verify if the function is correct
true_count = sum(outputs)
false_count = len(outputs) - true_count

print(f"\nTrue outputs: {true_count}")
print(f"False outputs: {false_count}")
print(f"\nFunction type: ", end="")

if f.is_constant:
    print(f"Constant (always returns {f.truth_table[0]})")
    # Verify all outputs are the same
    assert len(set(outputs)) == 1, "Error: constant function should return same value for all inputs"
    print("✓ Verified: All outputs are the same")
else:
    print("Balanced (returns True for exactly 8 inputs and False for 8 inputs)")
    # Verify there are exactly 8 True and 8 False outputs
    assert true_count == 8, "Error: balanced function should return True for exactly 8 inputs"
    print("✓ Verified: Exactly 8 True and 8 False outputs")

### Code Explanation

To verify our function works, we need to test it on all possible inputs. Since we have four Boolean inputs, that's $2^4 = 16$ different combinations to check.

We generate a random function and then systematically go through every combination of inputs using nested loops. For each combination, we call the function and store the result, printing it out in binary notation (using `0` for `False` and `1` for `True`) so it's easier to read.

Once we've tested all 16 inputs, we count how many returned `True` and how many returned `False`. Then we verify our function is working correctly: if it's constant, all 16 outputs should be identical, and if it's balanced, we should see exactly 8 `True` and 8 `False`. The assertions confirm this—they'll catch any mistakes and tell us if something went wrong.

# Problem 2: Classical Testing for Function Type
## Context
Deutsch's algorithm is designed to demonstrate a [potential advantage of quantum computing](https://www.quantamagazine.org/john-preskill-explains-quantum-supremacy-20191002/) compared to classical computation. We need to find out and understand the classical cost of solving this problem, so we can find out the advantage it gives.

In Problem 1, we generated random Boolean functions that are guaranteed to be either constant or balanced. We need to write a function that can **determine which type a given function is** by calling it and analysing the results.

### The Classical Problem
Given a function `f(a, b, c, d)` that takes 4 Boolean inputs and returns 1 Boolean output, we know it is either:
- **Constant**: returns the same value for all 16 possible input combinations
- **Balanced**: returns `True` for exactly 8 inputs and `False` for the other 8

The objective is to determine which type it is by calling the function and examining the results.

### Implementation
The function `determine_constant_balanced(f)` takes a function `f` as an input and determines whether it is constant or balanced.

#### Approach
1. Call the function `f` with different Boolean input combinations
2. Store the results we get
3. As soon as we see **both** `True` and `False` outputs, we know it is **balanced**
4. If we call all 16 combinations and see only one value, it is **constant**
5. Return the result as a string: `"constant"` or `"balanced"`

# Problem 3: Quantum Oracles

# Problem 4: Deutsch's Algorithm with Qiskit

# Problem 5: Scaling to the Deutsch–Jozsa Algorithm

# References
## Acknowledgements

# Conclusion

***
# End